# 課題5：映画レビューの評判分析

本課題ではAmazon傘下の「IMDb」に投稿された映画のレビュー（英語）を分析し、レビューがPositive（ポジティブ）か、Negative（ネガティブ）かの判別を行ないます。

データセットは、以下のサイトで配布されているものを利用します。

[Large Movie Review Dataset](https://ai.stanford.edu/%7Eamaas/data/sentiment/)

わからない場合は、ここまでのレッスン内容や各種ライブラリの公式ドキュメントを参照しましょう。

## 1. 必要なライブラリのimport

In [ ]:
# （変更しないでください）

# 必要なライブラリのimport
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# 文章ファイル検索用
import glob
import collections
from sklearn.feature_extraction import DictVectorizer

# DataFrameですべての列を表示する設定
pd.options.display.max_columns = None

# seabornによる装飾を適用する
sns.set_theme()

## 2. データの読み込み

In [3]:
# ダウンロードした圧縮ファイルを解凍する（変更しないでください）
import os
import tarfile
from pathlib import Path

base_dir = Path.cwd()
archive_candidates = [
    base_dir / 'aclImdb_v1.tar.gz',
    base_dir / 'Lesson09_ML-Practice' / 'aclImdb_v1.tar.gz',
]
archive_path = next((p for p in archive_candidates if p.exists()), None)

dataset_dir_candidates = [
    base_dir / 'aclImdb',
    base_dir / 'Lesson09_ML-Practice' / 'aclImdb',
]
dataset_dir = next((p for p in dataset_dir_candidates if p.exists()), None)

if dataset_dir is None and archive_path is not None:
    with tarfile.open(archive_path, 'r:gz') as tar:
        tar.extractall(base_dir)
    dataset_dir = next((p for p in dataset_dir_candidates if p.exists()), None)

if dataset_dir is None:
    raise FileNotFoundError('The IMDb dataset could not be found or extracted.')

os.chdir(dataset_dir.parent)
print(f'Using dataset directory: {dataset_dir}')

Using dataset directory: /Users/macuser/Downloads/aidev/Lesson09_ML-Practice/aclImdb


*./aclImdb* フォルダ内にあるファイルを読み込みます。

In [4]:
# trainフォルダのファイル一覧を取得（変更しないでください）
train_neg_files = glob.glob("./aclImdb/train/neg/*")
train_pos_files = glob.glob("./aclImdb/train/pos/*")

# testフォルダのファイル一覧を取得（変更しないでください）
test_neg_files = glob.glob("./aclImdb/test/neg/*")
test_pos_files = glob.glob("./aclImdb/test/pos/*")

In [5]:
# それぞれのファイル数を確認
import os
import glob
from pathlib import Path

base_dir = Path.cwd()
acl_dir = next((p for p in [base_dir / 'aclImdb', base_dir / 'Lesson09_ML-Practice' / 'aclImdb'] if p.exists()), None)

if acl_dir is None:
    raise FileNotFoundError('The aclImdb dataset directory was not found. Please run the extraction cell first.')

train_neg_files = glob.glob(str(acl_dir / 'train/neg/*'))
train_pos_files = glob.glob(str(acl_dir / 'train/pos/*'))
test_neg_files = glob.glob(str(acl_dir / 'test/neg/*'))
test_pos_files = glob.glob(str(acl_dir / 'test/pos/*'))

print('train_neg_files:', len(train_neg_files))
print('train_pos_files:', len(train_pos_files))
print('test_neg_files:', len(test_neg_files))
print('test_pos_files:', len(test_pos_files))

train_neg_files: 12500
train_pos_files: 12500
test_neg_files: 12500
test_pos_files: 12500


前処理をするため、合計50000あるファイルをリストにまとめます。

In [6]:
# ファイル名をまとめたリストを用意（変更しないでください）
filenames = train_neg_files + train_pos_files + test_neg_files + test_pos_files

# filenamesの長さを確認（変更しないでください）
len(filenames)

50000

リストの最初と最後のファイルを確認してみます。

In [7]:
# エンコーディング用定数（変更しないでください）
ENCODING = 'utf-8'

In [8]:
# 最初のファイルの内容を確認


In [9]:
# 最後のファイルの内容を確認


## 3. データの前処理

データの前処理として、形態素解析と行列への変換を行ないます。

### 形態素解析

In [16]:
# 文字列の中で使われている単語ごとの数を返す関数を作成
import collections


def count_words(text):
    words = text.lower().replace('\n', ' ').split()
    counter = collections.Counter(words)
    return counter

In [17]:
# 最初のファイルを使って、先ほど作成した関数をテスト
import glob
from pathlib import Path

base_dir = Path.cwd()
acl_dir = next((p for p in [base_dir / 'aclImdb', base_dir / 'Lesson09_ML-Practice' / 'aclImdb'] if p.exists()), None)

if acl_dir is None:
    raise FileNotFoundError('The aclImdb dataset directory was not found. Please run the extraction cell first.')

if 'filenames' not in globals():
    filenames = (
        glob.glob(str(acl_dir / 'train/neg/*'))
        + glob.glob(str(acl_dir / 'train/pos/*'))
        + glob.glob(str(acl_dir / 'test/neg/*'))
        + glob.glob(str(acl_dir / 'test/pos/*'))
    )

with open(filenames[0], 'r', encoding=ENCODING) as f:
    text = f.read()

count_words(text)

Counter({'to': 3,
         'the': 2,
         'film': 2,
         'a': 2,
         'working': 1,
         'with': 1,
         'one': 1,
         'of': 1,
         'best': 1,
         'shakespeare': 1,
         'sources,': 1,
         'this': 1,
         'manages': 1,
         'be': 1,
         'creditable': 1,
         "it's": 1,
         'source,': 1,
         'whilst': 1,
         'still': 1,
         'appealing': 1,
         'wider': 1,
         'audience.<br': 1,
         '/><br': 1,
         '/>branagh': 1,
         'steals': 1,
         'from': 1,
         'under': 1,
         "fishburne's": 1,
         'nose,': 1,
         'and': 1,
         "there's": 1,
         'talented': 1,
         'cast': 1,
         'on': 1,
         'good': 1,
         'form.': 1})

In [46]:
# 単語ごとの数のリストを作成（変更しないでください）
word_count_data = []
for filename in filenames:
    with open(filename, 'r', encoding=ENCODING) as f:
        text = f.read()
    word_count_data.append(count_words(text))

In [47]:
# すべてのファイルに対して、先ほど作成した関数を実行
# 上のセルですでに word_count_data を作成しているため、ここでは何もしません。
pass

In [48]:
# 単語ごとの数のリストの長さを確認
len(word_count_data)

50000

In [49]:
# 単語ごとの数のリストの0番目を表示
word_count_data[0]

Counter({'to': 3,
         'the': 2,
         'film': 2,
         'a': 2,
         'working': 1,
         'with': 1,
         'one': 1,
         'of': 1,
         'best': 1,
         'shakespeare': 1,
         'sources,': 1,
         'this': 1,
         'manages': 1,
         'be': 1,
         'creditable': 1,
         "it's": 1,
         'source,': 1,
         'whilst': 1,
         'still': 1,
         'appealing': 1,
         'wider': 1,
         'audience.<br': 1,
         '/><br': 1,
         '/>branagh': 1,
         'steals': 1,
         'from': 1,
         'under': 1,
         "fishburne's": 1,
         'nose,': 1,
         'and': 1,
         "there's": 1,
         'talented': 1,
         'cast': 1,
         'on': 1,
         'good': 1,
         'form.': 1})

### 行列への変換

In [50]:
# DictVectorizerを使用して行列に変換し、datasetに格納する
from sklearn.feature_extraction import DictVectorizer

vectorizer = DictVectorizer()
dataset = vectorizer.fit_transform(word_count_data)


In [51]:
# datasetの大きさを確認
dataset.shape

(50000, 390931)

In [52]:
# 各列に対応した単語を取得
vectorizer.get_feature_names_out()

array(['\x08\x08\x08\x08a', '\x10own', '!', ..., '₤250,000', '★★',
       '\uf0b7'], shape=(390931,), dtype=object)

## 4. 機械学習の実施

In [43]:
# 必要なライブラリの追加import（変更しないでください）
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

目的変数と説明変数を用意します。

In [ ]:
# 目的変数Yの用意
# neg12500 + pos12500 + neg12500 + pos12500 = 50000
import numpy as np

Y = np.array(
    [0] * len(train_neg_files)
    + [1] * len(train_pos_files)
    + [0] * len(test_neg_files)
    + [1] * len(test_pos_files)
)


In [53]:
# 上記のY、および前処理されたdatasetからデータを分割し、
# X_train, Y_train, X_test, Y_testに格納する
#
# 詳細：
#   - dataset を 50:50 で分割し、変数 X_train / X_test に代入
#   - 目的変数 Y も対応するように Y_train / Y_test に代入

from sklearn.model_selection import train_test_split

X_train, X_test, Y_train, Y_test = train_test_split(
    dataset,
    Y,
    test_size=0.5,
    random_state=42,
    stratify=Y,
)


In [54]:
# X_trainとY_trainを、train_test_splitで7:3に分割し、3割のほうを検証データ（X_valid, Y_valid）にする
X_train, X_valid, Y_train, Y_valid = train_test_split(
    X_train,
    Y_train,
    test_size=0.3,
    random_state=42,
    stratify=Y_train,
)


In [55]:
# ロジスティック回帰モデルを作成し、学習して、検証データによる予測を実施する
model = LogisticRegression(max_iter=1000)
model.fit(X_train, Y_train)

Y_pred_valid = model.predict(X_valid)

# classification_reportを実行し、検証データによるモデルの評価を行なう
print(classification_report(Y_valid, Y_pred_valid))

              precision    recall  f1-score   support

           0       0.59      0.59      0.59      3750
           1       0.59      0.58      0.59      3750

    accuracy                           0.59      7500
   macro avg       0.59      0.59      0.59      7500
weighted avg       0.59      0.59      0.59      7500



## 5. テストデータによる評価

最後に、テストデータで評価を行ないましょう。

In [57]:
# テストデータで予測を実施する
Y_pred_test = model.predict(X_test)

# classification_reportを実行し、テストデータによるモデルの評価を行なう
print(classification_report(Y_test, Y_pred_test))

              precision    recall  f1-score   support

           0       0.59      0.58      0.59     12500
           1       0.59      0.60      0.59     12500

    accuracy                           0.59     25000
   macro avg       0.59      0.59      0.59     25000
weighted avg       0.59      0.59      0.59     25000

